# CapitalIQ -- Rule-Based Finance & Investment Advisor Chatbot

**Project**  DecodeLabs AI Internship -- Project 1
**Author**  Ali Ahmad
**Batch**  2026
**Domain**  Finance & Investment Advisory

> *An LLM without rules is a hallucination engine. Today we build the skeleton that holds the intelligence.*
> -- DecodeLabs Architecture Briefing, Module 01

## Project Overview & Architecture
CapitalIQ is a deterministic, rule-based chatbot answering finance and investment questions across 16 intent categories. Unlike probabilistic LLMs, it is a **white-box** system: every response is fully traceable from input to output -- a legal requirement in regulated industries.

### The IPO Model
Raw Input -> Sanitize -> Match Intent -> Generate Response
(Stage I)    (Stage I)    (Stage II)      (Stage III)

### Why Dictionary over If-Elif?
An if-elif ladder is O(n) sequential scan. A Python dictionary is a hash map: **O(1) constant time** regardless of how many intents exist. Adding the 100th intent costs the same as adding the 5th.

### Two-Phase Architecture
- **Phase 1** -- The Logic Engine: complete chatbot on the core IPO model.
- **Phase 2** -- The Production Upgrade: confidence scoring, conversation memory, slot filling, and personalization.

---
# PHASE 1 -- The Logic Engine

Phase 1 builds the complete, working chatbot:
- Finance & Investment domain with **16 intents**
- O(1) dictionary-based knowledge base using `.get()`
- Input sanitization pipeline (`.lower().strip()` + regex)
- Multi-keyword intent matching for natural phrasing
- Clean fallback and graceful exit
- Full edge-case protection: empty input, whitespace, special characters

In [1]:
# CELL 4 -- Imports & Constants
# All Phase 1 code uses only the Python Standard Library.
# No third-party packages needed.

import re
from datetime import datetime

BOT_NAME    = 'CapitalIQ'
BOT_VERSION = '1.0.0'
AUTHOR      = 'Ali Ahmad'
BATCH       = '2026'

# Confidence threshold for Phase 2 matching
CONFIDENCE_THRESHOLD = 0.15

# Exit keywords -- triggers graceful session shutdown
EXIT_KEYWORDS = {'exit', 'quit', 'bye', 'goodbye', 'done', 'close'}

print(f'Bot: {BOT_NAME} v{BOT_VERSION} | Author: {AUTHOR} | Batch: {BATCH}')

Bot: CapitalIQ v1.0.0 | Author: Ali Ahmad | Batch: 2026


In [2]:
# CELL 5 -- Knowledge Base Dictionary (Phase 1)
# The knowledge base is a Python dict: intent_name -> response.
#
# WHY A DICTIONARY?
# O(1) constant-time lookup via hash map.
# An if-elif ladder with 16 branches is O(16) worst-case;
# at 1,000 intents it becomes O(1,000).
# The dict is always O(1) -- no contest.
#
# DOMAIN -- Finance & Investment Advisory:
# Finance has precise vocabulary with natural synonym clusters
# (btc/bitcoin/crypto), and responses must be hard-coded
# accurate -- no hallucination acceptable in finance.

KNOWLEDGE_BASE = {
    "greeting": (
        "Hello! I'm CapitalIQ, your personal Finance & Investment Advisor. "
        "I can help you understand stocks, bonds, mutual funds, ETFs, crypto, "
        "inflation, compound interest, diversification, risk, portfolio strategy, "
        "budgeting, emergency funds, SIPs, and IPOs. "
        "What financial topic would you like to explore today?"
    ),
    "goodbye": (
        "It was a pleasure advising you today! The best investment you can make "
        "is in yourself. Stay disciplined, keep learning, and let compound interest "
        "work in your favour. Goodbye and invest wisely!"
    ),
    "stocks": (
        "A stock (share or equity) represents fractional ownership in a company. "
        "Shareholders profit through capital appreciation and dividends. "
        "Stocks trade on exchanges like NYSE, NASDAQ, or BSE. They carry higher "
        "risk than bonds but historically deliver higher long-term returns. "
        "Key metrics: P/E ratio, EPS, revenue growth, debt-to-equity ratio."
    ),
    "bonds": (
        "A bond is a fixed-income debt instrument where you lend money to a "
        "government or corporation in exchange for periodic coupon payments and "
        "return of principal at maturity. Government bonds are the safest; "
        "corporate bonds offer higher yields but carry default risk. Bonds "
        "provide portfolio stability and hedge against equity volatility."
    ),
    "mutual_funds": (
        "A mutual fund pools capital from many investors to build a professionally "
        "managed, diversified portfolio. Each investor holds units proportional to "
        "their contribution. Key cost: the Expense Ratio -- annual fee charged by "
        "the fund house. Types: equity, debt, hybrid, and index funds. "
        "Always compare a fund's CAGR against its benchmark before investing."
    ),
    "etf": (
        "An ETF (Exchange-Traded Fund) trades on a stock exchange throughout the "
        "day like a regular stock. Most ETFs passively track an index such as "
        "Nifty 50 or S&P 500, resulting in very low expense ratios -- often below "
        "0.1%. ETFs are ideal for long-term, low-cost passive investors seeking "
        "broad market exposure without active stock-picking risk."
    ),
    "crypto": (
        "Cryptocurrency is a decentralised digital currency secured by cryptography "
        "and recorded on a blockchain. Bitcoin (BTC) and Ethereum (ETH) are the "
        "largest by market cap. Crypto offers high return potential but comes with "
        "extreme volatility and regulatory uncertainty. Never allocate more than "
        "5-10% of your portfolio to crypto, and only invest what you can afford "
        "to lose entirely."
    ),
    "inflation": (
        "Inflation is the rate at which the general price level rises over time, "
        "eroding purchasing power. Central banks target roughly 2% annual inflation. "
        "As an investor your returns must outpace inflation -- holding cash alone "
        "is insufficient. Asset classes that historically beat inflation include "
        "equities, real estate, gold, and inflation-linked bonds."
    ),
    "compound_interest": (
        "Compound interest is interest earned on both the original principal AND "
        "accumulated interest from previous periods. Formula: A = P(1 + r/n)^(nt). "
        "Starting early is the most powerful lever: a 25-year-old investing "
        "5,000/month at 12% CAGR will vastly outperform a 35-year-old investing "
        "10,000/month by retirement -- purely due to extra compounding years."
    ),
    "diversification": (
        "Diversification is the risk-management strategy of spreading investments "
        "across multiple uncorrelated asset classes, sectors, and geographies. "
        "When one asset class falls, others may hold steady -- smoothing overall "
        "volatility. A well-diversified portfolio includes domestic equities, "
        "international equities, bonds, real estate (REITs), gold, and cash."
    ),
    "risk": (
        "Investment risk is the probability that your actual return differs from "
        "your expected return. Key types: (1) Market Risk -- broad market declines; "
        "(2) Credit Risk -- issuer default; (3) Liquidity Risk -- cannot sell; "
        "(4) Inflation Risk -- returns beaten by inflation. Your risk tolerance "
        "shapes your asset allocation between equities and bonds."
    ),
    "portfolio": (
        "An investment portfolio is your complete collection of financial assets: "
        "stocks, bonds, ETFs, mutual funds, real estate, gold, and cash. "
        "Construction starts with your goals, time horizon, and risk tolerance. "
        "A classic 60/40 allocation (60% equities, 40% bonds) suits moderate "
        "investors. Rebalance annually to restore your target allocation."
    ),
    "budgeting": (
        "Budgeting is the foundation of financial success. Use the 50/30/20 Rule: "
        "50% of net income to Needs, 30% to Wants, 20% to Savings and Investments. "
        "Priority before investing: (1) build a 3-6 month emergency fund, "
        "(2) eliminate high-interest debt, then (3) invest aggressively."
    ),
    "emergency_fund": (
        "An emergency fund is 3-6 months of living expenses in a highly liquid, "
        "principal-safe account such as a high-yield savings account or liquid "
        "mutual fund. It prevents you from liquidating investments during a "
        "financial crisis. This is Step Zero of investing -- establish it BEFORE "
        "putting money in stocks or mutual funds."
    ),
    "sip": (
        "A SIP (Systematic Investment Plan) invests a fixed amount in a mutual "
        "fund at regular intervals, typically monthly. SIPs use Rupee-Cost "
        "Averaging: you buy more units when prices are low and fewer when high, "
        "reducing average cost and eliminating the need to time the market. "
        "Automate your SIP on salary day so investing happens first."
    ),
    "ipo": (
        "An IPO (Initial Public Offering) is when a private company lists on a "
        "stock exchange for the first time. IPOs can generate listing-day gains "
        "but carry significant risk: limited financial history, lock-up expiry "
        "selling pressure, and valuations driven by hype. Always read the DRHP "
        "(Draft Red Herring Prospectus) before applying."
    ),
}

FALLBACK_RESPONSE = (
    "I am not sure I understood that. I specialise in: stocks, bonds, "
    "mutual funds, ETFs, crypto, inflation, compound interest, "
    "diversification, risk, portfolios, budgeting, emergency funds, "
    "SIPs, and IPOs. Could you rephrase your question?"
)

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} intents loaded")
print(f"Intents: {list(KNOWLEDGE_BASE.keys())}")

Knowledge base: 16 intents loaded
Intents: ['greeting', 'goodbye', 'stocks', 'bonds', 'mutual_funds', 'etf', 'crypto', 'inflation', 'compound_interest', 'diversification', 'risk', 'portfolio', 'budgeting', 'emergency_fund', 'sip', 'ipo']


In [3]:
# CELL 6 -- Input Sanitization Function
# Stage I of the IPO model.
#
# WHY SANITIZE FIRST?
# Without sanitization, 'Bitcoin!', 'BITCOIN', '  bitcoin  ',
# each fail to match 'bitcoin' in the dict.
# Normalising before matching multiplies effective coverage
# without adding a single new intent.
#
# PIPELINE (5 stages):
#   1. Type guard  -- returns '' for None or non-string
#   2. Lowercase   -- 'BITCOIN' becomes 'bitcoin'
#   3. Strip       -- removes surrounding whitespace
#   4. Regex clean -- removes special chars (keeps ' and /)
#   5. Collapse    -- reduces multiple spaces to one

def sanitize(raw_input):
    """Sanitize raw user input through a 5-stage pipeline.

    Args:
        raw_input: Raw value from input(). Any type accepted.

    Returns:
        str: Clean normalised string. '' for blank or invalid input.
    """
    if not isinstance(raw_input, str):      # Stage 1: type guard
        return ""
    cleaned = raw_input.lower()             # Stage 2: lowercase
    cleaned = cleaned.strip()               # Stage 3: strip whitespace
    cleaned = re.sub(r"[^a-z0-9\s'/]", " ", cleaned)  # Stage 4: noise removal
    cleaned = re.sub(r"\s+", " ", cleaned).strip()     # Stage 5: collapse
    return cleaned


def is_empty(sanitized_input):
    """Return True if sanitized input is empty."""
    return len(sanitized_input) == 0


def tokenize(sanitized_input):
    """Split sanitized input into word tokens.

    Args:
        sanitized_input (str): Clean string from sanitize().

    Returns:
        list[str]: Word tokens. Empty list for empty input.
    """
    if not sanitized_input:
        return []
    return sanitized_input.split()


# -- Sanity tests
test_cases = [
    ("  BITCOIN!!  ",  "bitcoin"),
    ("What's a bond?", "what's a bond"),
    ("50/30/20 rule",  "50/30/20 rule"),
    ("   ",            ""),
    (None,             ""),
    ("!@#$%",          ""),
]
print("Sanitizer Tests:")
all_ok = True
for raw, expected in test_cases:
    result = sanitize(raw)
    ok = result == expected
    if not ok: all_ok = False
    print(f"  [{'PASS' if ok else 'FAIL'}]  {repr(str(raw)):<28} -> {repr(result)}")
print("All tests passed!" if all_ok else "SOME TESTS FAILED.")

Sanitizer Tests:
  [PASS]  '  BITCOIN!!  '              -> 'bitcoin'
  [PASS]  "What's a bond?"             -> "what's a bond"
  [PASS]  '50/30/20 rule'              -> '50/30/20 rule'
  [PASS]  '   '                        -> ''
  [PASS]  'None'                       -> ''
  [PASS]  '!@#$%'                      -> ''
All tests passed!


In [4]:
# CELL 7 -- Intent Matching Function (Phase 1)
# Stage II of the IPO model.
#
# TWO strategies in order:
#
# Strategy 1: Exact Synonym Lookup (O(1))
#   Maps known phrases to canonical intent names.
#   'btc' -> 'crypto', 'hi' -> 'greeting', 'sip' -> 'sip'
#
# Strategy 2: Multi-Keyword Token Voting
#   Each recognised token votes for an intent.
#   Intent with most votes wins.
#   Handles: "tell me how compound interest works"
#   -> tokens 'compound','interest' -> compound_interest

SYNONYM_MAP = {
    "hi": "greeting", "hello": "greeting", "hey": "greeting",
    "sup": "greeting", "help": "greeting", "start": "greeting",
    "good morning": "greeting", "good evening": "greeting",
    "bye": "goodbye", "goodbye": "goodbye", "quit": "goodbye",
    "exit": "goodbye", "done": "goodbye", "close": "goodbye",
    "stocks": "stocks", "stock": "stocks", "shares": "stocks",
    "share": "stocks", "equity": "stocks", "stock market": "stocks",
    "bond": "bonds", "bonds": "bonds", "fixed income": "bonds",
    "treasury": "bonds", "debenture": "bonds", "gsec": "bonds",
    "mutual fund": "mutual_funds", "mutual funds": "mutual_funds",
    "fund": "mutual_funds", "nav": "mutual_funds",
    "etf": "etf", "etfs": "etf", "index fund": "etf",
    "passive investing": "etf", "exchange traded fund": "etf",
    "crypto": "crypto", "bitcoin": "crypto", "btc": "crypto",
    "ethereum": "crypto", "eth": "crypto", "blockchain": "crypto",
    "nft": "crypto", "defi": "crypto", "cryptocurrency": "crypto",
    "altcoin": "crypto",
    "inflation": "inflation", "deflation": "inflation",
    "cpi": "inflation", "purchasing power": "inflation",
    "compound interest": "compound_interest",
    "compounding": "compound_interest", "compound": "compound_interest",
    "power of compounding": "compound_interest",
    "time value of money": "compound_interest",
    "diversification": "diversification", "diversify": "diversification",
    "asset allocation": "diversification",
    "risk": "risk", "volatility": "risk",
    "risk tolerance": "risk", "investment risk": "risk",
    "portfolio": "portfolio", "holdings": "portfolio",
    "rebalancing": "portfolio",
    "budget": "budgeting", "budgeting": "budgeting",
    "50/30/20": "budgeting", "personal finance": "budgeting",
    "money management": "budgeting",
    "emergency fund": "emergency_fund", "savings": "emergency_fund",
    "emergency savings": "emergency_fund",
    "rainy day fund": "emergency_fund",
    "sip": "sip", "systematic investment plan": "sip",
    "monthly investment": "sip", "rupee cost averaging": "sip",
    "ipo": "ipo", "initial public offering": "ipo",
    "public listing": "ipo", "drhp": "ipo",
}

KEYWORD_INTENT_MAP = {
    "hi": "greeting", "hello": "greeting", "hey": "greeting",
    "bye": "goodbye", "exit": "goodbye", "quit": "goodbye",
    "stock": "stocks", "stocks": "stocks", "share": "stocks",
    "shares": "stocks", "equity": "stocks", "dividend": "stocks",
    "bond": "bonds", "bonds": "bonds", "coupon": "bonds",
    "treasury": "bonds", "yield": "bonds",
    "mutual": "mutual_funds", "fund": "mutual_funds",
    "nav": "mutual_funds", "aum": "mutual_funds",
    "etf": "etf", "etfs": "etf", "index": "etf", "passive": "etf",
    "crypto": "crypto", "bitcoin": "crypto", "btc": "crypto",
    "ethereum": "crypto", "eth": "crypto",
    "blockchain": "crypto", "nft": "crypto", "defi": "crypto",
    "inflation": "inflation", "deflation": "inflation",
    "cpi": "inflation", "purchasing": "inflation",
    "compound": "compound_interest", "compounding": "compound_interest",
    "interest": "compound_interest", "principal": "compound_interest",
    "diversification": "diversification", "diversify": "diversification",
    "allocation": "diversification", "hedge": "diversification",
    "risk": "risk", "volatility": "risk", "drawdown": "risk",
    "portfolio": "portfolio", "holdings": "portfolio",
    "rebalancing": "portfolio", "rebalance": "portfolio",
    "budget": "budgeting", "budgeting": "budgeting",
    "expenses": "budgeting", "income": "budgeting", "spending": "budgeting",
    "emergency": "emergency_fund", "savings": "emergency_fund",
    "liquid": "emergency_fund", "rainy": "emergency_fund",
    "sip": "sip", "systematic": "sip", "monthly": "sip", "recurring": "sip",
    "ipo": "ipo", "listing": "ipo", "prospectus": "ipo", "drhp": "ipo",
}


def is_exit_command(sanitized_input):
    """Check if sanitized input is an exit/quit command.

    Args:
        sanitized_input (str): Clean input string.

    Returns:
        bool: True if session should terminate.
    """
    if sanitized_input in EXIT_KEYWORDS:
        return True
    return any(t in EXIT_KEYWORDS for t in tokenize(sanitized_input))


def match_intent_p1(sanitized_input):
    """Phase 1 matcher: synonym lookup -> keyword voting -> fallback.

    Args:
        sanitized_input (str): Clean input from sanitize().

    Returns:
        str: Response from KNOWLEDGE_BASE, or FALLBACK_RESPONSE.
    """
    if not sanitized_input:
        return "I did not catch that. What finance topic can I help with?"

    # Strategy 1: Exact synonym lookup -- O(1) hash map
    if sanitized_input in SYNONYM_MAP:
        intent = SYNONYM_MAP[sanitized_input]
        return KNOWLEDGE_BASE.get(intent, FALLBACK_RESPONSE)

    # Strategy 2: Multi-keyword token voting
    tokens = tokenize(sanitized_input)
    votes = {}
    for token in tokens:
        if token in KEYWORD_INTENT_MAP:
            intent = KEYWORD_INTENT_MAP[token]
            votes[intent] = votes.get(intent, 0) + 1

    if not votes:
        return FALLBACK_RESPONSE   # FALLBACK -- spec requirement 4

    best = max(votes, key=lambda k: votes[k])
    return KNOWLEDGE_BASE.get(best, FALLBACK_RESPONSE)


# -- Quick tests
tests = [
    ("btc",                       "crypto"),
    ("what is compound interest", "compound_interest"),
    ("how do stocks work",        "stocks"),
    ("tell me about sip",         "sip"),
    ("purple dancing unicorns",   None),
]
print("Matcher Tests:")
for inp, exp_key in tests:
    result = match_intent_p1(sanitize(inp))
    if exp_key:
        ok = result == KNOWLEDGE_BASE[exp_key]
    else:
        ok = result == FALLBACK_RESPONSE
    label = exp_key if exp_key else "FALLBACK"
    print(f"  [{'PASS' if ok else 'FAIL'}]  '{inp}' -> {label}")
print("Phase 1 matcher ready.")

Matcher Tests:
  [PASS]  'btc' -> crypto
  [PASS]  'what is compound interest' -> compound_interest
  [PASS]  'how do stocks work' -> stocks
  [PASS]  'tell me about sip' -> sip
  [PASS]  'purple dancing unicorns' -> FALLBACK
Phase 1 matcher ready.


In [5]:
# CELL 8 -- Response Generation Function (Phase 1)
# Stage III of the IPO model: Output.
#
# generate_response_p1() orchestrates the pipeline and handles
# the edge case the matcher never sees: completely empty input.
# Phase 2 will expand this with personalization and memory.

def generate_response_p1(raw_input):
    """Full Phase 1 IPO pipeline for one user input.

    Pipeline: sanitize() -> is_empty() guard -> match_intent_p1()

    Handles all edge cases: empty, whitespace, special chars, None.
    None of these will ever crash the loop.

    Args:
        raw_input (str): Raw text from input().

    Returns:
        str: Bot response string ready to print.
    """
    # INPUT stage
    clean = sanitize(raw_input)

    # GUARD: empty or whitespace -- friendly prompt, no crash
    if is_empty(clean):
        return (
            "It looks like you sent an empty message. "
            "Ask me about stocks, bonds, crypto, SIPs, or any finance topic!"
        )

    # PROCESS + OUTPUT stage
    return match_intent_p1(clean)


# -- Validate the full Phase 1 pipeline
demo = [
    ("hello",                           "greeting"),
    ("What is Bitcoin?",                "crypto"),
    ("Tell me about compound interest", "compound_interest"),
    ("     ",                           "empty"),
    ("!@#$%^&*()",                      "empty"),
    ("xyzzy foobar baz",                "fallback"),
]

print("Full Pipeline Tests (sanitize -> match -> response):")
print("=" * 58)
for user_in, label in demo:
    resp = generate_response_p1(user_in)
    print(f"\n  Input    : {repr(user_in)[:55]}")
    print(f"  Label    : [{label}]")
    r = resp[:92] + "..." if len(resp) > 92 else resp
    print(f"  Response : {r}")
print("\nPhase 1 response pipeline validated.")

Full Pipeline Tests (sanitize -> match -> response):

  Input    : 'hello'
  Label    : [greeting]
  Response : Hello! I'm CapitalIQ, your personal Finance & Investment Advisor. I can help you understand ...

  Input    : 'What is Bitcoin?'
  Label    : [crypto]
  Response : Cryptocurrency is a decentralised digital currency secured by cryptography and recorded on a...

  Input    : 'Tell me about compound interest'
  Label    : [compound_interest]
  Response : Compound interest is interest earned on both the original principal AND accumulated interest...

  Input    : '     '
  Label    : [empty]
  Response : It looks like you sent an empty message. Ask me about stocks, bonds, crypto, SIPs, or any fi...

  Input    : '!@#$%^&*()'
  Label    : [empty]
  Response : It looks like you sent an empty message. Ask me about stocks, bonds, crypto, SIPs, or any fi...

  Input    : 'xyzzy foobar baz'
  Label    : [fallback]
  Response : I am not sure I understood that. I specialise in: stocks, b

In [6]:
# CELL 9 -- Main Chatbot Loop (Phase 1)
# The HEARTBEAT of the chatbot -- the while True loop.
#
# IPO Model in the loop:
#   INPUT   -> input() captures raw user text each iteration
#   PROCESS -> generate_response_p1() sanitizes and matches
#   OUTPUT  -> print() displays the response
#
# The loop runs until the kill command (exit/quit/bye).
# KeyboardInterrupt (Ctrl+C) is caught gracefully.
#
# NOTE: Uncomment run_phase1() at the bottom to chat live.
#       Cell 10 runs an automated demo without blocking.

def run_phase1():
    """Launch the Phase 1 interactive chatbot loop.

    The while True loop is the heartbeat. It keeps CapitalIQ
    alive until the kill command breaks the loop cleanly.
    """
    border = "=" * 60
    print(f"\n{border}")
    print(f"  {BOT_NAME} v{BOT_VERSION}  --  Finance Advisor (Phase 1)")
    print(f"  DecodeLabs AI  |  Batch {BATCH}  |  {AUTHOR}")
    print(f"{border}")
    print("  Type 'exit' or 'quit' to end the session.")
    print(f"{border}\n")

    # INPUT LOOP -- satisfies DecodeLabs spec requirement 1
    while True:
        try:
            raw_input = input("  You: ")   # INPUT stage
        except (EOFError, KeyboardInterrupt):
            print(f"\n  {BOT_NAME}: Session interrupted. Goodbye!")
            break

        # EXIT CHECK -- before any processing
        if is_exit_command(sanitize(raw_input)):
            farewell = generate_response_p1(raw_input)
            print(f"\n  {BOT_NAME}: {farewell}\n")
            break  # EXIT STRATEGY -- spec requirement 5

        # PROCESS + OUTPUT
        response = generate_response_p1(raw_input)
        print(f"\n  {BOT_NAME}: {response}\n")


# Uncomment to run interactively:
# run_phase1()
print("Phase 1 loop defined.")
print("Uncomment run_phase1() above to start an interactive session.")
print("Cell 10 runs an automated 10-case demo without blocking.")

Phase 1 loop defined.
Uncomment run_phase1() above to start an interactive session.
Cell 10 runs an automated 10-case demo without blocking.


In [7]:
# CELL 10 -- Automated Demo / Test Runner (Phase 1)
# 10 pre-written test inputs run automatically.
#
# Coverage:
#  1.  Greeting           -- exact synonym match
#  2.  Known intent (SIP) -- multi-keyword natural sentence
#  3.  Exit keyword       -- graceful exit detection
#  4.  Partial match      -- compound interest via keywords
#  5.  Synonym (BTC)      -- btc to crypto
#  6.  Empty/whitespace   -- edge case
#  7.  Special chars only -- edge case
#  8.  Long natural input -- noisy sentence
#  9.  Known intent       -- inflation query
#  10. Unknown/fallback   -- pure noise

TEST_INPUTS = [
    ("hello",                                                        "Greeting -- exact synonym"),
    ("how does a sip work",                                          "Known intent -- multi-keyword"),
    ("quit",                                                         "Exit keyword -- kill command"),
    ("tell me about compound interest please",                       "Partial natural phrasing"),
    ("btc",                                                          "Synonym -- btc to crypto"),
    ("     ",                                                        "Edge case -- whitespace only"),
    ("!@#$%^&*()",                                                   "Edge case -- special chars"),
    ("I have been hearing about ipo lately should i invest in it",   "Long natural sentence"),
    ("what is inflation and how does it affect savings",             "Known intent -- multi-keyword"),
    ("purple dancing elephants in zero gravity",                     "Unknown input -- expect fallback"),
]

print("=" * 68)
print("  CapitalIQ Phase 1 -- Automated Demo (10 Test Cases)")
print("=" * 68)

for i, (user_in, label) in enumerate(TEST_INPUTS, 1):
    clean = sanitize(user_in)
    if is_exit_command(clean):
        tag = "[EXIT]"
    elif is_empty(clean):
        tag = "[EMPTY]"
    elif match_intent_p1(clean) == FALLBACK_RESPONSE:
        tag = "[FALLBACK]"
    else:
        tag = "[MATCH]"
    response = generate_response_p1(user_in)
    print(f"\n  Test {i:02d}  {tag}  --  {label}")
    print(f"  You  : {repr(user_in[:65])}")
    r = response[:115] + "..." if len(response) > 115 else response
    print(f"  Bot  : {r}")
    print(f"  {'-'*62}")

print("\nAll 10 Phase 1 test cases completed.")

  CapitalIQ Phase 1 -- Automated Demo (10 Test Cases)

  Test 01  [MATCH]  --  Greeting -- exact synonym
  You  : 'hello'
  Bot  : Hello! I'm CapitalIQ, your personal Finance & Investment Advisor. I can help you understand stocks, bonds, mutual f...
  --------------------------------------------------------------

  Test 02  [MATCH]  --  Known intent -- multi-keyword
  You  : 'how does a sip work'
  Bot  : A SIP (Systematic Investment Plan) invests a fixed amount in a mutual fund at regular intervals, typically monthly....
  --------------------------------------------------------------

  Test 03  [EXIT]  --  Exit keyword -- kill command
  You  : 'quit'
  Bot  : It was a pleasure advising you today! The best investment you can make is in yourself. Stay disciplined, keep learn...
  --------------------------------------------------------------

  Test 04  [MATCH]  --  Partial natural phrasing
  You  : 'tell me about compound interest please'
  Bot  : Compound interest is interest earne

# PHASE 2 -- The Production Upgrade
Phase 2 is a **strict superset** of Phase 1. Every Phase 1 feature works unchanged.

### Architectural change

Phase 1 `match_intent_p1()` returns `str`.
Phase 2 `match_intent_p2()` returns a **3-tuple** `(intent_name, response, confidence)`.

In [8]:
# CELL 12 -- Enhanced Knowledge Base: Synonym Coverage Report
# Phase 2 uses KNOWLEDGE_BASE and SYNONYM_MAP from Cells 5 and 7.
# This cell audits their coverage.

alias_counts = {}
for phrase, intent in SYNONYM_MAP.items():
    alias_counts[intent] = alias_counts.get(intent, 0) + 1

samples = {}
for phrase, intent in SYNONYM_MAP.items():
    if intent not in samples:
        samples[intent] = phrase

print("Phase 2 -- Synonym/Alias Coverage Report")
print("=" * 56)
print(f"  {'Intent':<22}  {'Aliases':>7}  Sample Trigger")
print("  " + "-" * 52)

for intent in KNOWLEDGE_BASE:
    count  = alias_counts.get(intent, 0)
    sample = samples.get(intent, "--")
    print(f"  {intent:<22}  {count:>7}  '{sample}'")

total = sum(alias_counts.values())
print("  " + "-" * 52)
print(f"  {'TOTAL':<22}  {total:>7}  aliases across {len(KNOWLEDGE_BASE)} intents")
print("=" * 56)
print(f"\nVerified: {len(KNOWLEDGE_BASE)} intents, {total} aliases.")

Phase 2 -- Synonym/Alias Coverage Report
  Intent                  Aliases  Sample Trigger
  ----------------------------------------------------
  greeting                      8  'hi'
  goodbye                       6  'bye'
  stocks                        6  'stocks'
  bonds                         6  'bond'
  mutual_funds                  4  'mutual fund'
  etf                           5  'etf'
  crypto                       10  'crypto'
  inflation                     4  'inflation'
  compound_interest             5  'compound interest'
  diversification               3  'diversification'
  risk                          4  'risk'
  portfolio                     3  'portfolio'
  budgeting                     5  'budget'
  emergency_fund                4  'emergency fund'
  sip                           4  'sip'
  ipo                           4  'ipo'
  ----------------------------------------------------
  TOTAL                        81  aliases across 16 intents

Verified: 16 i

In [9]:
# CELL 13 -- Confidence Scoring Function (Phase 2)
# Upgrades binary match/no-match to a graduated score [0, 1].
#
# HOW IT WORKS:
#   1. Tokenize the input.
#   2. Count keyword votes per intent.
#   3. Divide by total token count -> normalised score.
#   4. Pick the highest-scoring intent.
#   5. If score < CONFIDENCE_THRESHOLD -> return fallback.
#
# WHY NORMALISE?
# A 20-token sentence always outscores a 3-token sentence in
# raw votes even if the shorter query is more focused.
# Dividing by total tokens puts all queries on the same scale.

def score_intents(tokens):
    """Compute normalised confidence scores for candidate intents.

    Args:
        tokens (list[str]): Tokenised sanitised user input.

    Returns:
        dict[str, float]: intent -> score in [0.0, 1.0].
    """
    if not tokens:
        return {}
    raw_votes = {}
    for token in tokens:
        if token in KEYWORD_INTENT_MAP:
            intent = KEYWORD_INTENT_MAP[token]
            raw_votes[intent] = raw_votes.get(intent, 0) + 1
    if not raw_votes:
        return {}
    total = len(tokens)
    return {k: round(v / total, 3) for k, v in raw_votes.items()}


def match_intent_p2(sanitized_input):
    """Phase 2 matcher: synonym lookup -> scoring -> threshold gate.

    Returns a 3-tuple so memory can log structured metadata.

    Args:
        sanitized_input (str): Clean normalised input.

    Returns:
        tuple[str, str, float]: (intent, response, confidence)
    """
    if not sanitized_input:
        return ("none", "I did not catch that. Ask me a finance question!", 0.0)

    # Strategy 1: Exact synonym -> confidence = 1.0
    if sanitized_input in SYNONYM_MAP:
        intent = SYNONYM_MAP[sanitized_input]
        return (intent, KNOWLEDGE_BASE.get(intent, FALLBACK_RESPONSE), 1.0)

    # Strategy 2: Confidence scoring from token votes
    tokens = tokenize(sanitized_input)
    scores = score_intents(tokens)

    if not scores:
        return ("fallback", FALLBACK_RESPONSE, 0.0)

    best  = max(scores, key=lambda k: scores[k])
    score = scores[best]

    # Threshold gate: reject low-confidence matches
    if score < CONFIDENCE_THRESHOLD:
        return ("fallback", FALLBACK_RESPONSE, score)

    return (best, KNOWLEDGE_BASE.get(best, FALLBACK_RESPONSE), score)


# -- Visualise confidence scores
queries = [
    "bitcoin",
    "tell me about compound interest",
    "what is the difference between stocks and bonds",
    "purple unicorns dance in space",
    "sip",
    "I want to understand how to diversify my portfolio",
]
print("Confidence Scoring Visualisation")
print("=" * 65)
print(f"  {'Query':<44} {'Intent':<18} Score  Bar")
print("  " + "-" * 61)
for q in queries:
    intent, _, score = match_intent_p2(sanitize(q))
    bar = "#" * int(score * 20)
    print(f"  {q[:43]:<44} {intent:<18} {score:.3f}  {bar}")
print("=" * 65)
print(f"\nPhase 2 confidence scorer ready. Threshold = {CONFIDENCE_THRESHOLD}")

Confidence Scoring Visualisation
  Query                                        Intent             Score  Bar
  -------------------------------------------------------------
  bitcoin                                      crypto             1.000  ####################
  tell me about compound interest              compound_interest  0.400  ########
  what is the difference between stocks and b  fallback           0.125  ##
  purple unicorns dance in space               fallback           0.000  
  sip                                          sip                1.000  ####################
  I want to understand how to diversify my po  fallback           0.111  ##

Phase 2 confidence scorer ready. Threshold = 0.15


In [10]:
# CELL 14 -- Conversation Memory Module (Phase 2)
# A stateless chatbot treats every message in isolation.
# ConversationMemory gives CapitalIQ state awareness:
#
#   1. History Log  -- ordered record of every turn with
#      user message, bot response, intent, confidence.
#      Displayed as a full transcript at session end.
#
#   2. Slot Store   -- key-value dict for named entities.
#      In production NLP (Rasa, LangChain), this is called
#      slot filling or context tracking.

class ConversationMemory:
    """Session-scoped memory store for CapitalIQ.

    Attributes:
        history    (list[dict]): All conversation turns.
        slots      (dict):       Named entity / context store.
        turn_count (int):        Number of completed turns.
    """

    def __init__(self):
        """Initialise fresh empty memory for a new session."""
        self.history    = []
        self.slots      = {}
        self.turn_count = 0

    def add_turn(self, user_input, bot_response, intent="unknown", confidence=0.0):
        """Record one completed turn with full metadata."""
        self.turn_count += 1
        self.history.append({
            "turn":       self.turn_count,
            "timestamp":  datetime.now().strftime("%H:%M:%S"),
            "user":       user_input,
            "bot":        bot_response,
            "intent":     intent,
            "confidence": round(confidence, 3),
        })
        self.set_slot("last_intent", intent)

    def set_slot(self, key, value):
        """Store a named entity in the slot store."""
        self.slots[key] = str(value)

    def get_slot(self, key, default=""):
        """Retrieve a slot value; returns default if not set."""
        return self.slots.get(key, default)

    def has_slot(self, key):
        """True if the slot exists and is non-empty."""
        return key in self.slots and bool(self.slots[key])

    def extract_name(self, sanitized_input):
        """Extract user name from introductory patterns.

        Patterns: 'my name is X' | 'i am X' | "i'm X" | 'call me X'

        Returns:
            str or None: Title-cased name, or None if not found.
        """
        for pattern in [
            r"my name is (\w+)",
            r"\bi am (\w+)",
            r"\bi'm (\w+)",
            r"call me (\w+)",
        ]:
            m = re.search(pattern, sanitized_input)
            if m:
                return m.group(1).title()
        return None

    def display_history(self):
        """Print the full session transcript to stdout."""
        if not self.history:
            print("  No conversation history.")
            return
        sep = "=" * 62
        print(f"\n{sep}")
        print("  SESSION TRANSCRIPT  --  CapitalIQ")
        print(sep)
        for t in self.history:
            preview = t["bot"][:70] + "..." if len(t["bot"]) > 70 else t["bot"]
            print(f"\n  Turn {t['turn']} @ {t['timestamp']}")
            print(f"  You : {t['user']}")
            print(f"  Bot : {preview}")
            print(f"  Intent: {t['intent']}  |  Confidence: {t['confidence']}")
        print(f"\n{sep}")
        print(f"  Total turns: {self.turn_count}")
        print(f"{sep}\n")

    def get_summary(self):
        """Return session statistics as a dict."""
        if not self.history:
            return {"total_turns": 0, "unique_intents": [], "most_discussed": None}
        skip = {"fallback", "none", "unknown", "goodbye", "empty"}
        counts = {}
        for t in self.history:
            if t["intent"] not in skip:
                counts[t["intent"]] = counts.get(t["intent"], 0) + 1
        return {
            "total_turns":    self.turn_count,
            "unique_intents": list(set(t["intent"] for t in self.history if t["intent"] not in skip)),
            "user_name":      self.get_slot("user_name") or "Unknown",
            "most_discussed": max(counts, key=lambda k: counts[k]) if counts else None,
        }

    def reset(self):
        """Clear all state -- useful for testing."""
        self.history = []
        self.slots   = {}
        self.turn_count = 0


# -- Demonstrate the memory module
mem = ConversationMemory()
mem.add_turn("hello",          "Hello! I am CapitalIQ...", "greeting", 1.0)
mem.add_turn("my name is Ali", "Nice to meet you, Ali!",  "greeting", 0.8)
mem.add_turn("what is bitcoin","Cryptocurrency is...",     "crypto",   0.333)
mem.add_turn("explain sip",    "A SIP invests fixed...",   "sip",      1.0)

print("ConversationMemory Demonstration")
print("-" * 40)
print(f"  Name extracted : {mem.extract_name('my name is ali')}")
print(f"  user_name slot : '{mem.get_slot('user_name', 'Not set')}'")
print(f"  last_intent    : '{mem.get_slot('last_intent')}'")
print(f"  Turn count     : {mem.turn_count}")
print()
mem.display_history()
print(f"Summary: {mem.get_summary()}")
print("ConversationMemory module validated.")

ConversationMemory Demonstration
----------------------------------------
  Name extracted : Ali
  user_name slot : 'Not set'
  last_intent    : 'sip'
  Turn count     : 4


  SESSION TRANSCRIPT  --  CapitalIQ

  Turn 1 @ 15:02:57
  You : hello
  Bot : Hello! I am CapitalIQ...
  Intent: greeting  |  Confidence: 1.0

  Turn 2 @ 15:02:57
  You : my name is Ali
  Bot : Nice to meet you, Ali!
  Intent: greeting  |  Confidence: 0.8

  Turn 3 @ 15:02:57
  You : what is bitcoin
  Bot : Cryptocurrency is...
  Intent: crypto  |  Confidence: 0.333

  Turn 4 @ 15:02:57
  You : explain sip
  Bot : A SIP invests fixed...
  Intent: sip  |  Confidence: 1.0

  Total turns: 4

Summary: {'total_turns': 4, 'unique_intents': ['greeting', 'crypto', 'sip'], 'user_name': 'Unknown', 'most_discussed': 'greeting'}
ConversationMemory module validated.


In [11]:
# CELL 15 -- Full Upgraded Chatbot Loop (Phase 2 -- Production)
# Phase 2 is a strict superset of Phase 1. All Phase 1 features
# work unchanged. Phase 2 adds:
#   * Confidence-scored matching (match_intent_p2)
#   * Conversation memory and slot filling (ConversationMemory)
#   * Name extraction and personalisation
#   * Session transcript display on exit
#
# Uncomment run_phase2() for interactive mode.

def generate_response_p2(raw_input, memory):
    """Full Phase 2 IPO pipeline with memory integration.

    Args:
        raw_input (str):              Raw text from input().
        memory (ConversationMemory):  Active session memory.

    Returns:
        tuple[str, str, float]: (response, intent, confidence)
    """
    clean = sanitize(raw_input)

    # GUARD: empty input
    if is_empty(clean):
        msg = "Empty message. Ask me about stocks, crypto, SIPs, or budgeting!"
        memory.add_turn(raw_input, msg, "empty", 0.0)
        return (msg, "empty", 0.0)

    # PROCESS: Phase 2 confidence-scored matching
    intent, response, confidence = match_intent_p2(clean)

    # PERSONALISATION: name extraction on first mention
    extracted = memory.extract_name(clean)
    if extracted:
        memory.set_slot("user_name", extracted)
        response = f"Nice to meet you, {extracted}! {response}"

    # PERSONALISATION: use stored name periodically (every 4th turn)
    elif (memory.has_slot("user_name")
          and intent not in ("fallback", "none", "goodbye")
          and memory.turn_count > 0
          and memory.turn_count % 4 == 0):
        name = memory.get_slot("user_name")
        response += f"\n\n  Great question, {name}! Keep exploring."

    # OUTPUT: record in memory
    memory.add_turn(raw_input, response, intent, confidence)
    return (response, intent, confidence)


def run_phase2():
    """Launch the Phase 2 interactive loop (production version)."""
    mem = ConversationMemory()
    border = "=" * 62
    print(f"\n{border}")
    print("  CapitalIQ v2.0.0  --  Finance & Investment Advisor")
    print(f"  DecodeLabs AI  |  Batch 2026  |  Ali Ahmad")
    print(f"{border}")
    print("  Topics: stocks, bonds, ETFs, crypto, SIPs, IPOs")
    print("  Type 'exit' or 'quit' to end and view your transcript.")
    print(f"{border}\n")

    while True:   # THE HEARTBEAT -- Phase 2 version
        try:
            raw_input = input("  You: ")
        except (EOFError, KeyboardInterrupt):
            print("\n  CapitalIQ: Session interrupted. Goodbye!")
            break

        if is_exit_command(sanitize(raw_input)):
            resp, _, _ = generate_response_p2(raw_input, mem)
            print(f"\n  CapitalIQ: {resp}\n")
            mem.display_history()
            name = mem.get_slot("user_name") or "Investor"
            print(f"  Thanks, {name}! Invest wisely. Goodbye!\n")
            break

        resp, intent, conf = generate_response_p2(raw_input, mem)
        print(f"\n  CapitalIQ: {resp}")
        print(f"  [Confidence: {conf:.3f} | Intent: {intent}]\n")


# -- Phase 2 Automated Simulation
sim_mem = ConversationMemory()
sim_inputs = [
    "hello", "my name is Ali", "what is compound interest",
    "explain diversification", "btc", "how does a sip work",
    "what should i know about ipo", "purple dinosaurs",
    "tell me about budgeting", "bye",
]

print("=" * 68)
print("  CapitalIQ Phase 2 -- Automated Simulation (10 Turns)")
print("=" * 68)

for user_in in sim_inputs:
    if is_exit_command(sanitize(user_in)):
        resp, intent, conf = generate_response_p2(user_in, sim_mem)
        print(f"\n  [EXIT] You  : {user_in!r}")
        r = resp[:100] + "..." if len(resp) > 100 else resp
        print(f"         Bot  : {r}")
        break
    resp, intent, conf = generate_response_p2(user_in, sim_mem)
    print(f"\n  You     : {user_in!r}")
    print(f"  Intent  : {intent}  (confidence: {conf:.3f})")
    r = resp[:105] + "..." if len(resp) > 105 else resp
    print(f"  Bot     : {r}")

sim_mem.display_history()

# Uncomment for interactive mode:
# run_phase2()
print("Phase 2 complete. Uncomment run_phase2() for interactive mode.")

  CapitalIQ Phase 2 -- Automated Simulation (10 Turns)

  You     : 'hello'
  Intent  : greeting  (confidence: 1.000)
  Bot     : Hello! I'm CapitalIQ, your personal Finance & Investment Advisor. I can help you understand stocks, bonds...

  You     : 'my name is Ali'
  Intent  : fallback  (confidence: 0.000)
  Bot     : Nice to meet you, Ali! I am not sure I understood that. I specialise in: stocks, bonds, mutual funds, ETF...

  You     : 'what is compound interest'
  Intent  : compound_interest  (confidence: 0.500)
  Bot     : Compound interest is interest earned on both the original principal AND accumulated interest from previou...

  You     : 'explain diversification'
  Intent  : diversification  (confidence: 0.500)
  Bot     : Diversification is the risk-management strategy of spreading investments across multiple uncorrelated ass...

  You     : 'btc'
  Intent  : crypto  (confidence: 1.000)
  Bot     : Cryptocurrency is a decentralised digital currency secured by cryptography 